<a href="https://colab.research.google.com/github/zydanne-costa/Ondas_ADCP_SCO_Mar_Nov_2025/blob/main/Wave_Wind_1h.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configuração do Ambiente

In [1]:
import locale

# Define o locale para português do Brasil, que usa vírgula como separador decimal
# Isso afeta a formatação de números em alguns contextos, incluindo matplotlib quando configurado.
# Certifique-se de que o locale 'pt_BR.UTF-8' esteja disponível no ambiente.
try:
    locale.setlocale(locale.LC_NUMERIC, 'pt_BR.UTF-8')
except locale.Error:
    print("Locale 'pt_BR.UTF-8' not found. Using default locale. Decimal separator might not be comma.")

Locale 'pt_BR.UTF-8' not found. Using default locale. Decimal separator might not be comma.


Esta célula configura o local (locale) para português do Brasil, o que pode influenciar a formatação de números, especialmente para separadores decimais (vírgula em vez de ponto).

## Montagem do Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Esta célula monta o Google Drive no ambiente do Colab, permitindo o acesso aos arquivos armazenados no Drive.

## Importação de Dados de Onda

In [3]:
# importando dados de ONDA do periodo chuvoso e menos chuvoso
import pandas as pd

# chuvoso
file_path = '/content/drive/MyDrive/Ondas/Dados/Refinados/WaveData_1H_CHU.txt'
df_wdata_chu_1h = pd.read_csv(file_path, sep='\t')

# menos chuvoso
file_path = '/content/drive/MyDrive/Ondas/Dados/Refinados/WaveData_1H_SEC.txt'
df_wdata_sec_1h = pd.read_csv(file_path, sep='\t')


Esta célula importa os dados de onda (Altura Significativa e Período Significativo) para os períodos chuvoso e menos chuvoso a partir de arquivos `.txt`.

## Importação de Dados de Vento

In [4]:
# importando dados de VENTO do periodo chuvoso e menos chuvoso

# chuvoso
file_path = '/content/drive/MyDrive/Ondas/Dados/Refinados/WindData_1H_CHU.txt'
df_winddata_chu_1h = pd.read_csv(file_path, sep='\t')

# menos chuvoso
file_path = '/content/drive/MyDrive/Ondas/Dados/Refinados/WindData_1H_SEC.txt'
df_winddata_sec_1h = pd.read_csv(file_path, sep='\t')


Esta célula importa os dados de vento (velocidade e direção) para os períodos chuvoso e menos chuvoso a partir de arquivos `.txt`.

## Importação de Dados de Profundidade

In [6]:
# chuvoso
file_path_depth_chu = '/content/drive/MyDrive/Ondas/Dados/Refinados/Depth_CHU.txt'
df_depth_chu = pd.read_csv(file_path_depth_chu, sep='\t')
df_depth_chu['DataHora'] = pd.to_datetime(df_depth_chu['Tempo'])

# menos chuvoso
file_path_depth_sec = '/content/drive/MyDrive/Ondas/Dados/Refinados/Depth_SEC.txt'
df_depth_sec = pd.read_csv(file_path_depth_sec, sep='\t')
df_depth_sec['DataHora'] = pd.to_datetime(df_depth_sec['Tempo'])

Esta célula importa os dados de profundidade para os períodos chuvoso e menos chuvoso e converte a coluna 'Tempo' para o formato datetime.

## Fusão de Dados de Onda e Vento (Período Chuvoso)

In [7]:
# Merge de df_wdata_chu_1h and df_winddata_chu_1h
wwdata_chu = pd.merge(
    df_wdata_chu_1h,
    df_winddata_chu_1h,
    left_on='DataHora',
    right_on='datetime',
    how='left'  # Alterado para 'left' merge
).drop(columns=['datetime'], errors='ignore') # Adicionado errors='ignore' caso 'datetime' já tenha sido removido

Esta célula realiza a fusão (merge) dos DataFrames de dados de onda e vento para o período chuvoso, usando 'DataHora' e 'datetime' como chaves e removendo a coluna 'datetime'.

## Fusão de Dados de Onda e Vento (Período Menos Chuvoso)

In [8]:
# Merge de df_wdata_sec_1h and df_winddata_sec_1h
wwdata_sec = pd.merge(
    df_wdata_sec_1h,
    df_winddata_sec_1h,
    left_on='DataHora',
    right_on='datetime',
    how='inner'
).drop(columns=['datetime'])

Esta célula realiza a fusão (merge) dos DataFrames de dados de onda e vento para o período menos chuvoso, usando 'DataHora' e 'datetime' como chaves e removendo a coluna 'datetime'.

## Fusão de Dados de Profundidade com Onda e Vento

In [9]:
import pandas as pd

# Convert DataHora columns to datetime before merging
wwdata_chu['DataHora'] = pd.to_datetime(wwdata_chu['DataHora'])
wwdata_sec['DataHora'] = pd.to_datetime(wwdata_sec['DataHora'])

# Merge para o período chuvoso
wwddata_chu = pd.merge(
    wwdata_chu,
    df_depth_chu[['DataHora', 'Depth_m']],
    on='DataHora',
    how='left'
)

# Merge para o período menos chuvoso
wwddata_sec = pd.merge(
    wwdata_sec,
    df_depth_sec[['DataHora', 'Depth_m']],
    on='DataHora',
    how='left'
)

Esta célula faz a fusão dos dados de profundidade (`df_depth_chu` e `df_depth_sec`) com os DataFrames combinados de onda e vento (`wwdata_chu` e `wwdata_sec`), usando a coluna `DataHora`.

## Renomear Colunas de Vento (Período Chuvoso e Menos Chuvoso)

In [22]:
# Renomear colunas de vento no DataFrame wwddata_chu
wwddata_chu = wwddata_chu.rename(columns={'vel_ms': 'Wd_vel_ms', 'dir_deg': 'Wd_dir_deg'})

# Renomear colunas de vento no DataFrame wwddata_sec
wwddata_sec = wwddata_sec.rename(columns={'vel_ms': 'Wd_vel_ms', 'dir_deg': 'Wd_dir_deg'})

Esta célula renomeia as colunas de velocidade e direção do vento no DataFrame `wwddata_chu` para nomes mais descritivos (`Wd_vel_ms` e `Wd_dir_deg`).

## Exibir Amostra dos DataFrames Mesclados

In [24]:
print('First 5 rows of wwddata_chu:')
display(wwddata_chu.head())
print('\nFirst 5 rows of wwddata_sec:')
display(wwddata_sec.head())

First 5 rows of wwddata_chu:


,DataHora,Hs,Ts,Hmax,Tmax,Hmed,Tmed,Nondas,Wd_vel_ms,Wd_dir_deg,Depth_m
0,2025-03-27 13:00:00,0.42,4.64,0.75,5.0,0.14,3.81,293,5.44,44.6,NaN
1,2025-03-27 14:00:00,0.54,4.25,1.00,10.0,0.45,4.13,584,5.14,38.7,2.191
2,2025-03-27 15:00:00,0.84,3.71,1.27,5.0,0.52,2.84,2153,4.90,35.4,2.856
3,2025-03-27 16:00:00,0.57,3.40,1.00,9.0,0.40,2.92,2086,4.09,35.7,3.791
4,2025-03-27 17:00:00,0.65,3.46,1.30,2.5,0.41,2.83,2157,3.70,50.1,4.619



First 5 rows of wwddata_sec:


,DataHora,Hs,Ts,Hmax,Tmax,Hmed,Tmed,Nondas,Wd_dir_deg,Wd_vel_ms,Depth_m
0,2025-11-22 18:00:00,0.94,3.61,1.10,8.0,0.70,2.77,737,19.1,7.24,3.941
1,2025-11-22 19:00:00,0.61,3.28,1.03,4.0,0.44,2.69,754,22.3,7.38,4.612
2,2025-11-22 20:00:00,0.87,3.91,1.37,3.0,0.66,2.99,676,22.4,7.60,5.098
3,2025-11-22 21:00:00,0.48,2.91,0.73,2.5,0.36,2.52,808,21.4,7.93,5.371
4,2025-11-22 22:00:00,0.52,3.44,0.92,6.0,0.37,2.68,759,23.8,8.59,5.287


Esta célula exibe as primeiras 5 linhas dos DataFrames `wwddata_chu` e `wwddata_sec` após as fusões e renomeações, para verificar a estrutura dos dados.

## Importação de Bibliotecas para Visualização e Conversão de DataHora

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

# Converter a coluna 'DataHora' para o tipo datetime em ambos os DataFrames
wwdata_chu['DataHora'] = pd.to_datetime(wwdata_chu['DataHora'])
wwdata_sec['DataHora'] = pd.to_datetime(wwdata_sec['DataHora'])

Esta célula importa as bibliotecas `matplotlib.pyplot` e `seaborn` para visualização de dados e garante que a coluna 'DataHora' em `wwdata_chu` e `wwdata_sec` esteja no formato datetime, essencial para análise temporal.

## Preparação Adicional de DataHora para Visualização

In [12]:
# Converter a coluna 'DataHora' para o tipo datetime em ambos os DataFrames
wwdata_chu['DataHora'] = pd.to_datetime(wwdata_chu['DataHora'])
wwdata_sec['DataHora'] = pd.to_datetime(wwdata_sec['DataHora'])

Esta célula assegura novamente que a coluna 'DataHora' em `wwdata_chu` e `wwdata_sec` esteja no formato datetime. Esta etapa é crucial para qualquer visualização ou análise que dependa de datas e horas.

## Exportação dos Dados Combinados (Período Chuvoso)

In [14]:
# Exportar wwdata_chu para um arquivo .txt
output_file_chu = '/content/drive/MyDrive/Ondas/Dados/Refinados/wwdata_chu.txt'
wwddata_chu.to_csv(output_file_chu, sep='\t', index=False)
print(f"'wwdata_chu.txt' exportado para {output_file_chu}")

'wwdata_chu.txt' exportado para /content/drive/MyDrive/Ondas/Dados/Refinados/wwdata_chu.txt


Esta célula exporta o DataFrame `wwddata_chu` (com dados de onda, vento e profundidade renomeados) para um arquivo de texto no formato `.txt`.

## Exportação dos Dados Combinados (Período Menos Chuvoso)

In [15]:
# Exportar wwdata_sec para um arquivo .txt
output_file_sec = '/content/drive/MyDrive/Ondas/Dados/Refinados/wwdata_sec.txt'
wwddata_sec.to_csv(output_file_sec, sep='\t', index=False)
print(f"'wwdata_sec.txt' exportado para {output_file_sec}")

'wwdata_sec.txt' exportado para /content/drive/MyDrive/Ondas/Dados/Refinados/wwdata_sec.txt


Esta célula exporta o DataFrame `wwddata_sec` (com dados de onda, vento e profundidade renomeados) para um arquivo de texto no formato `.txt`.